# 02 : Préparation des données

**Entrée :** Table `payments_raw` (données brutes sauvegardées par le notebook 01)  
**Sortie :** Tables `payments` (paiements nettoyés) et `hcp_features` (profils + cible `retenu`)

---

## Objectif

Appliquer les **décisions de préparation** prises à l'issue de l'exploration (Notebook 01). On repart du brut déjà téléchargé (`payments_raw`), on nettoie, on agrège au niveau du professionnel de santé, et on construit la cible de rétention.

## Rappel des décisions (Notebook 01)

1. **Dédoublonnage :** Supprimer les lignes strictement identiques (1 204 lignes, artefacts de réinjection identifiés en EDA).
2. **Typage :** Convertir les montants (`total_amount_of_payment_usdollars`) et l'année (`program_year`) en formats numériques (`float` et `int`).
3. **Spécialité :** Extraire le niveau haut de la spécialité médicale en découpant sur le séparateur `|` (`covered_recipient_specialty_1`).
4. **Harmonisation :** Normaliser les noms de laboratoires (`.str.upper().str.strip()`) pour garantir la fiabilité du comptage des partenaires et des parts de marché.
5. **Périmètre HCP :** Exclure les lignes sans identifiant de professionnel (notamment les hôpitaux universitaires - *Covered Recipient Teaching Hospital*).
6. **Agrégation :** Structurer la table au niveau individuel (profil d'engagement de l'année 2022) : montant total, moyen **et médian**, diversité de laboratoires/natures.
7. **Part en valeur :** Calculer la part de chaque nature de paiement dans le **montant** (pas dans le simple volume de paiements).
8. **Cible :** Construire la variable binaire `retenu` (présence avérée d'un paiement en 2023).

## 0. Chargement des données brutes

On repart de `payments_raw` écrit par le notebook 01 (aucun re-téléchargement).

In [1]:
import sqlite3, pathlib
import pandas as pd

ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd().parent / "data").exists() else pathlib.Path.cwd()
DB_PATH = ROOT / "data" / "openpayments.sqlite"
with sqlite3.connect(DB_PATH) as con:
    raw = pd.read_sql("SELECT * FROM payments_raw", con)
print("Fichier Brut chargé :", raw.shape)

Fichier Brut chargé : (186426, 9)


## 1. Nettoyage (décisions 1 à 3)

Conversion des montants et de l'année en numérique ; extraction du niveau haut de la spécialité ; exclusion des lignes sans identifiant de professionnel.

In [2]:
import pandas as pd

def clean(df):
    df = df.copy()
    
    # --- 1. IDENTIFIANTS MANQUANTS ---
    taille_initiale = len(df)
    df = df[df["covered_recipient_profile_id"].notna()]
    df = df[df["covered_recipient_profile_id"].astype(str).str.len() > 0]
    nb_ids_supprimes = taille_initiale - len(df)
    
    # Conversions numériques
    df["total_amount_of_payment_usdollars"] = pd.to_numeric(df["total_amount_of_payment_usdollars"], errors="coerce")
    df["program_year"] = pd.to_numeric(df["program_year"], errors="coerce").astype("Int64")
    
    # --- 2. SÉPARATEURS (Spécialités) ---
    spe_avant = df["covered_recipient_specialty_1"].copy()
    df["specialty"] = df["covered_recipient_specialty_1"].fillna("Unknown").str.split("|").str[0].str.strip()
    # On compare en gérant les NaN pour voir combien de valeurs ont été réellement modifiées
    nb_spe_modifiees = (spe_avant.fillna("Unknown") != df["specialty"]).sum()
    
    # --- 3. NOMS STANDARDISÉS (Laboratoires) ---
    labo_avant = df["applicable_manufacturer_or_applicable_gpo_making_payment_name"].copy()
    df["applicable_manufacturer_or_applicable_gpo_making_payment_name"] = (
        df["applicable_manufacturer_or_applicable_gpo_making_payment_name"].str.upper().str.strip()
    )
    nb_labos_modifies = (labo_avant.fillna("") != df["applicable_manufacturer_or_applicable_gpo_making_payment_name"].fillna("")).sum()
    
    # --- 4. DOUBLONS ---
    nb_doublons = df.duplicated().sum()
    df = df.drop_duplicates()
    
    # --- IMPRESSION DES LOGS DE NETTOYAGE ---
    print("RAPPORT DE NETTOYAGE :")
    print(f"  - {nb_ids_supprimes} lignes supprimées (identifiant manquant/vide)")
    print(f"  - {nb_spe_modifiees} spécialités extraites (valeurs modifiées ou NaN remplacés)")
    print(f"  - {nb_labos_modifies} noms de laboratoires standardisés (casse/espaces corrigés)")
    print(f"  - {nb_doublons} lignes dédoublonnées supprimées")
    print("-" * 40)
    
    return df

# Application de la fonction
payments = clean(raw)
print("Après nettoyage :", payments.shape, "|", payments["covered_recipient_profile_id"].nunique(), "professionnels")

RAPPORT DE NETTOYAGE :
  - 203 lignes supprimées (identifiant manquant/vide)
  - 186223 spécialités extraites (valeurs modifiées ou NaN remplacés)
  - 151975 noms de laboratoires standardisés (casse/espaces corrigés)
  - 1212 lignes dédoublonnées supprimées
----------------------------------------
Après nettoyage : (185011, 10) | 7753 professionnels


Le nettoyage s'est déroulé en plusieurs étapes tracées pour garantir la qualité de la donnée :
* **Exclusion de 203 lignes** ne possédant pas d'identifiant de professionnel (valeurs manquantes ou vides).
* **Typage** : Conversion des montants et de l'année en format numérique.
* **Extraction de la spécialité** : Récupération du niveau haut de la spécialité (avant le séparateur `|`) et traitement des valeurs nulles sur les **186 223** lignes restantes.
* **Standardisation** : Correction de la casse et suppression des espaces sur **151 975** noms de laboratoires pour unifier les entités.
* **Dédoublonnage** : Suppression finale de **1 212 lignes** redondantes.

**Après nettoyage : (185 011, 10) | 7753 professionnels**

> **Observation.** L'exclusion des 203 lignes sans identifiant retire notamment les paiements faits aux hôpitaux universitaires ou mal renseignés ; il ne reste que les paiements aux professionnels individuels. L'effort massif de standardisation sur la casse des laboratoires (plus de 151 000 corrections) a permis de faire émerger et d'éliminer efficacement les doublons cachés (1 212 lignes retirées au total), garantissant qu'aucun paiement n'est comptabilisé deux fois. Le jeu est désormais parfaitement propre et prêt pour l'agrégation.

## 2. Agrégation au niveau du professionnel + cible de rétention (décisions 4 et 5)

Pour chaque professionnel, on résume son engagement **en 2022** ; la cible `retenu` vaut 1 s'il reçoit au moins un paiement **en 2023**.

In [3]:
def build_features(df_feat, ids_target):
    g = df_feat.groupby("covered_recipient_profile_id")
    feats = pd.DataFrame({
        "n_payments": g.size(),
        "total_amount": g["total_amount_of_payment_usdollars"].sum(),
        "mean_amount": g["total_amount_of_payment_usdollars"].mean(),
        "median_amount": g["total_amount_of_payment_usdollars"].median(),
        "n_manufacturers": g["applicable_manufacturer_or_applicable_gpo_making_payment_name"].nunique(),
        "n_natures": g["nature_of_payment_or_transfer_of_value"].nunique(),
        "specialty": g["specialty"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "Unknown"),
        "state": g["recipient_state"].first(),
    })
    # Part de la VALEUR (montant) de chaque nature de paiement, pas du volume
    for nature, col in [("Food and Beverage", "share_food"), ("Travel and Lodging", "share_travel"),
                        ("Consulting Fee", "share_consulting"),
                        ("Compensation for services other than consulting, including serving as faculty or as a speaker at a venue other than a continuing education program", "share_speaker"),
                        ("Education", "share_education")]:
        part = df_feat[df_feat["nature_of_payment_or_transfer_of_value"] == nature].groupby("covered_recipient_profile_id")["total_amount_of_payment_usdollars"].sum()
        feats[col] = (part / feats["total_amount"]).reindex(feats.index).fillna(0.0)
    feats["retenu"] = feats.index.to_series().isin(ids_target).astype(int)
    return feats.reset_index()

df_2022 = payments[payments["program_year"] == 2022]
ids_2023 = set(payments.loc[payments["program_year"] == 2023, "covered_recipient_profile_id"].unique())
features = build_features(df_2022, ids_2023)
print("Profils 2022 :", len(features), "| taux de retention :", round(features["retenu"].mean(), 3))
features.head()

Profils 2022 : 5707 | taux de retention : 0.733


,covered_recipient_profile_id,n_payments,total_amount,mean_amount,median_amount,n_manufacturers,n_natures,specialty,state,share_food,share_travel,share_consulting,share_speaker,share_education,retenu
0,1000304,10,176.44,17.644000,13.845,2,1,Allopathic & Osteopathic Physicians,WV,1.000000,0.000000,0.0,0.0,0.0,1
1,1001595,14,2533.93,180.995000,63.845,3,2,Allopathic & Osteopathic Physicians,WV,0.209852,0.790148,0.0,0.0,0.0,1
2,100261,78,1199.01,15.371923,13.275,17,1,Allopathic & Osteopathic Physicians,WV,1.000000,0.000000,0.0,0.0,0.0,1
3,1002721,55,896.83,16.306000,16.680,17,1,Allopathic & Osteopathic Physicians,WV,1.000000,0.000000,0.0,0.0,0.0,1
4,1003026,6,108.56,18.093333,17.240,5,1,Allopathic & Osteopathic Physicians,WV,1.000000,0.000000,0.0,0.0,0.0,1


**Observation.** On obtient un profil par professionnel de 2022 avec sa cible de rétention (~73 %). Le passage au calcul de la part par *valeur financière* (plutôt que par volume de transactions) reflète de manière beaucoup plus réaliste l'enjeu économique de la relation entre le laboratoire et le professionnel (par exemple, un seul gros paiement de consulting pèsera désormais plus lourd que 10 petits repas). Le jeu est prêt pour lamodélisation (notebook 03).

## 3. Écriture de la base finale

In [4]:
with sqlite3.connect(DB_PATH) as con:
    payments.to_sql("payments", con, if_exists="replace", index=False)
    features.to_sql("hcp_features", con, if_exists="replace", index=False)
print("Base ecrite :", DB_PATH)
print("  table payments     :", len(payments), "lignes")
print("  table hcp_features  :", len(features), "profils")

Base ecrite : c:\Juliette Vanessa\Desktop\notebook\pharma-commercial-genai\data\openpayments.sqlite
  table payments     : 185011 lignes
  table hcp_features  : 5707 profils


**Observation.** La base de données contient désormais deux tables propres : *payments* (qui conserve la granularité de chaque transaction pour d'éventuelles requêtes analytiques) et *hcp_features** (la table agrégée au niveau du professionnel, avec ses variables descriptives et sa cible de rétention). Le dataset est prêt : prochaine étape, Notebook 03 pour l'entraînement des modèles prédictifs.